<a href="https://colab.research.google.com/github/Farhan-ANWAR0611/Distributed-Machine-Learning/blob/main/Copy_of_Predictive_Modeling_for_Banking_Trends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Predictive Modeling for Banking Trends**

##### **Name**          - Nischay Verma

## **GitHub Link** :
https://github.com/Nischay-verma/Capstone-2-ML-Engineering-Financial-Forecasting-Frontier-Distributed-ML

## **Drive Link** :
https://drive.google.com/drive/folders/1lv1MivgxVPQHjG78N2A_v2d2VKHM4Kvg?usp=sharing




###**1: Data Loading and Initial Exploration**

 #### Load the bank.csv dataset into a Spark DataFrame.

In [ ]:
# Import SparkSession
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("ML_Spark_Bank").getOrCreate()

# Load the dataset
df = spark.read.csv("/content/bank (1).csv", header=True, inferSchema=True)

# Show the schema
df.printSchema()

# Show the first 5 rows
df.show(5)


root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)

+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|        job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+

The printSchema() output displays the structure of the DataFrame and confirms that Spark has correctly inferred the data types of all columns. For example, columns such as age, balance, day, and duration are recognized as integer types, while job, marital, education, and other categorical attributes are identified as string types. This validation ensures that the dataset is properly loaded and ready for further analysis and machine learning tasks.

The df.show(5) command displays the first five records of the dataset, allowing a quick visual inspection of the data. It helps verify that the values have been loaded correctly, the columns contain valid information, and the target variable y (subscription status) is present and properly populated. This step is essential for confirming data quality before performing transformations and model training.

#### Perform basic exploratory data analysis (EDA) to understand the dataset. Display the schema and the first few rows.

In [ ]:
#ques 2 Perform basic exploratory data analysis (EDA) to understand the dataset. Display the schema,
#the first few rows, the number of rows and columns, and descriptive statistics for numeric columns.


# Already done:
df.printSchema()
df.show(5)

# New EDA: Row and column count
print(f"Number of Rows: {df.count()}")
print(f"Number of Columns: {len(df.columns)}")

# Summary statistics for numeric columns
df.describe(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']).show()



root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)

+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|        job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+

The df.count() and len(df.columns) commands provide the total number of records and columns in the dataset. For example, the dataset contains 4,521 rows and 17 columns, confirming that the data has been loaded successfully and is ready for analysis and machine learning tasks.

The df.describe().show() function generates summary statistics for numerical features such as age, balance, day, duration, campaign, pdays, and previous. It displays key metrics including count, mean, standard deviation, minimum, and maximum values. These statistics help in understanding the data distribution, identifying potential outliers, and assessing feature variability. For instance, the average balance indicates typical customer account balances, while a high maximum duration or large standard deviation may suggest the presence of extreme values that require further investigation.

###**2: Data Preprocessing**

#### Handle missing values in the dataset.


In [ ]:
# ques 3 Handle Missing Values in the Dataset


# Import functions
from pyspark.sql.functions import col, when, count

# Count missing (null) values in each column
missing_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
missing_counts.show()


+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
|age|job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
|  0|  0|      0|        0|      0|      0|      0|   0|      0|  0|    0|       0|       0|    0|       0|       0|  0|
+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+



The dataset contains no missing values, so no data cleaning or imputation is required. It is clean and ready for preprocessing and analysis.

#### Handle Outliers in the Dataset

In [ ]:
#ques 4 Handle Outliers in the Dataset

# Function to detect outliers using IQR
def detect_outliers_iqr(df, column):
    quantiles = df.approxQuantile(column, [0.25, 0.75], 0.05)
    Q1 = quantiles[0]
    Q3 = quantiles[1]
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print(f"\nOutlier range for '{column}': [{lower_bound}, {upper_bound}]")
    outlier_count = df.filter((col(column) < lower_bound) | (col(column) > upper_bound)).count()
    print(f"Number of outliers in '{column}': {outlier_count}")

# Check for outliers in key numeric columns
for col_name in ['age', 'balance', 'duration']:
    detect_outliers_iqr(df, col_name)



Outlier range for 'age': [12.0, 68.0]
Number of outliers in 'age': 67

Outlier range for 'balance': [-1630.0, 2914.0]
Number of outliers in 'balance': 647

Outlier range for 'duration': [-176.0, 576.0]
Number of outliers in 'duration': 457


Outlier analysis identified 67 unusual age values, 647 extreme balance values, and 457 long-duration calls. While balance and duration outliers may represent genuine customer behavior, the duration feature may be excluded from modeling to avoid data leakage.

#### Convert categorical variables to numerical format using StringIndexer or OneHotEncoder.


In [ ]:
# ques5 Convert categorical variables to numerical format using StringIndexer or OneHotEncoder.

from pyspark.ml.feature import StringIndexer

# List of categorical columns
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'y']

# Create indexers
indexers = [StringIndexer(inputCol=col, outputCol=col + "_indexed") for col in categorical_cols]

# Apply transformations
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=indexers)
df_indexed = pipeline.fit(df).transform(df)

# Show result of indexed columns
df_indexed.select([col + "_indexed" for col in categorical_cols] + ["age", "balance", "duration"]).show(5)


+-----------+---------------+-----------------+---------------+---------------+------------+---------------+-------------+----------------+---------+---+-------+--------+
|job_indexed|marital_indexed|education_indexed|default_indexed|housing_indexed|loan_indexed|contact_indexed|month_indexed|poutcome_indexed|y_indexed|age|balance|duration|
+-----------+---------------+-----------------+---------------+---------------+------------+---------------+-------------+----------------+---------+---+-------+--------+
|        8.0|            0.0|              2.0|            0.0|            1.0|         0.0|            0.0|          8.0|             0.0|      0.0| 30|   1787|      79|
|        4.0|            0.0|              0.0|            0.0|            0.0|         1.0|            0.0|          0.0|             1.0|      0.0| 33|   4789|     220|
|        0.0|            1.0|              1.0|            0.0|            0.0|         0.0|            0.0|          5.0|             1.0|      

Categorical variables were converted into numeric values using StringIndexer. A Spark ML Pipeline was used to apply all indexers efficiently in a single step. This transformation is necessary because Spark ML algorithms require numeric input features. The target variable y was also indexed and used as the label for model training.

###**3: Feature Engineering and Data Transformation**

#### Create a feature vector using VectorAssembler by combining all feature columns.

In [ ]:
#ques 6 Create a feature vector using VectorAssembler by combining all feature columns.

from pyspark.ml.feature import VectorAssembler

# Define input features: numeric columns + indexed categorical columns (exclude target)
input_features = [
    'age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous',
    'job_indexed', 'marital_indexed', 'education_indexed', 'default_indexed',
    'housing_indexed', 'loan_indexed', 'contact_indexed',
    'month_indexed', 'poutcome_indexed'
]

# Assemble all input features into a single 'features' column
assembler = VectorAssembler(inputCols=input_features, outputCol="features")
final_df = assembler.transform(df_indexed)

# Select only the final feature vector and label
final_df.select("features", "y_indexed").show(5, truncate=False)


+--------------------------------------------------------------------------+---------+
|features                                                                  |y_indexed|
+--------------------------------------------------------------------------+---------+
|[30.0,1787.0,19.0,79.0,1.0,-1.0,0.0,8.0,0.0,2.0,0.0,1.0,0.0,0.0,8.0,0.0]  |0.0      |
|[33.0,4789.0,11.0,220.0,1.0,339.0,4.0,4.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0]|0.0      |
|[35.0,1350.0,16.0,185.0,1.0,330.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,5.0,1.0]|0.0      |
|[30.0,1476.0,3.0,199.0,4.0,-1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,3.0,0.0]  |0.0      |
|(16,[0,2,3,4,5,7,13],[59.0,5.0,226.0,1.0,-1.0,1.0,1.0])                   |0.0      |
+--------------------------------------------------------------------------+---------+
only showing top 5 rows


VectorAssembler was used to combine all selected input features into a single features vector, which is required by Spark ML algorithms. The transformed dataset now contains a features column with all predictor variables and a y_indexed column representing the numeric target label, making the data ready for model training and evaluation.

###**4: Model Training and Selection**

#### Choose a classification model (e.g., Logistic Regression, Decision Tree Classifier) for predicting the subscription to a term deposit.
#### Split the data into training and test sets.
#### Train the model on the training dataset.


In [ ]:
#ques 7 Choose a classification model (e.g., Logistic Regression, Decision Tree Classifier) for predicting the subscription to a term deposit.
#Split the data into training and test sets.
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Rename target column to 'label'
model_df = final_df.select("features", col("y_indexed").alias("label"))

# Split into training and test sets (80/20)
train_data, test_data = model_df.randomSplit([0.8, 0.2], seed=42)

# Initialize Logistic Regression model
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Train the model
lr_model = lr.fit(train_data)

# Make predictions on test data
predictions = lr_model.transform(test_data)

# Show predictions
predictions.select("features", "label", "prediction", "probability").show(5, truncate=False)



+-----------------------------------------------------------------------+-----+----------+-----------------------------------------+
|features                                                               |label|prediction|probability                              |
+-----------------------------------------------------------------------+-----+----------+-----------------------------------------+
|(16,[0,1,2,3,4,5,6,7,15],[24.0,299.0,6.0,209.0,1.0,321.0,1.0,3.0,1.0]) |0.0  |0.0       |[0.9593690832670528,0.040630916732947164]|
|(16,[0,1,2,3,4,5,6,7,15],[29.0,228.0,11.0,12.0,8.0,342.0,9.0,1.0,1.0]) |0.0  |0.0       |[0.9870533167274181,0.01294668327258186] |
|(16,[0,1,2,3,4,5,6,7,15],[30.0,-28.0,18.0,284.0,2.0,371.0,1.0,1.0,1.0])|0.0  |0.0       |[0.9511480590312332,0.04885194096876677] |
|(16,[0,1,2,3,4,5,6,7,15],[33.0,43.0,14.0,332.0,2.0,358.0,2.0,1.0,1.0]) |0.0  |0.0       |[0.9404157451105136,0.05958425488948638] |
|(16,[0,1,2,3,4,5,6,7,15],[36.0,23.0,8.0,152.0,2.0,347.0,1.0,2.0,1.0]

I chose Logistic Regression because it's effective for binary classification (predicting yes/no).

I split the dataset into:

80% training data

20% testing data

I renamed y_indexed to label (required by Spark ML).

The model was trained on the training data and used to predict the labels for test data.

The output shows:

prediction: 0.0 = No, 1.0 = Yes

probability: Confidence level for both classes



###**5: Model Evaluation**

#### Evaluate the model on the test dataset using appropriate metrics.


In [ ]:
# ques 8 Evaluate the model on the test dataset using appropriate metrics (Accuracy, Precision, Recall, F1 Score).
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Evaluator for Accuracy
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predictions)

# Evaluator for Precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)

# Evaluator for Recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)

# Evaluator for F1 Score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1 = evaluator_f1.evaluate(predictions)

# Display all metrics
print(f"🔍 Accuracy: {accuracy:.4f}")
print(f"🔍 Precision: {precision:.4f}")
print(f"🔍 Recall: {recall:.4f}")
print(f"🔍 F1 Score: {f1:.4f}")



🔍 Accuracy: 0.8917
🔍 Precision: 0.8707
🔍 Recall: 0.8917
🔍 F1 Score: 0.8707


The Logistic Regression model achieved an accuracy of 89.17%, indicating that it correctly classified the majority of client subscription outcomes. The precision score of 87.07% shows that when the model predicts a client will subscribe, the prediction is usually correct. The recall value of 89.17% demonstrates the model's ability to identify most actual subscribers, minimizing missed opportunities. Additionally, the F1-score of 87.07% reflects a strong balance between precision and recall, making the model reliable for predicting term deposit subscriptions. Overall, the results indicate that the model performs effectively with a high level of accuracy and consistency.


###**6: Hyperparameter Tuning**




#### Perform hyperparameter tuning (using ParamGridBuilder and CrossValidator.)


In [ ]:
# ques 9 Perform hyperparameter tuning using ParamGridBuilder and CrossValidator.
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Define evaluator
evaluator = BinaryClassificationEvaluator(labelCol="label")

# Build parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

# CrossValidator setup
cv = CrossValidator(estimator=lr,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=5)

# Fit model with cross-validation
cv_model = cv.fit(train_data)

# Evaluate the best model
best_model = cv_model.bestModel
best_predictions = best_model.transform(test_data)

# Recalculate Accuracy
best_accuracy = evaluator_acc.evaluate(best_predictions)
print(f"🔍 Best Model Accuracy after Tuning: {best_accuracy:.4f}")



🔍 Best Model Accuracy after Tuning: 0.8906


After applying 5-fold cross-validation for hyperparameter tuning, the best Logistic Regression model achieved an accuracy of 89.06%. This result is very close to the original model's accuracy of 89.17%, indicating that the initial hyperparameter settings were already near optimal. The consistency in performance demonstrates that the model is stable and generalizes well across different parameter configurations. Although the improvement is minimal, hyperparameter tuning is valuable as it validates the robustness and reliability of the model.

###**7: Advanced Analysis**

#### Analyze the feature importances (if applicable) or coefficients of the model to gain insights into which features are most influential in predicting the outcome.





In [ ]:
import pandas as pd
# Convert Spark SparseVector to a Python list
coef_values = list(best_model.coefficients)

# Get feature names from the input_features list
features_list = input_features

# Create a DataFrame using feature names and their corresponding coefficient values
coef_df = pd.DataFrame({
    "Feature": features_list,
    "Coefficient": coef_values
}).sort_values(by="Coefficient", ascending=False)

# Print model intercept
print("Intercept:", best_model.intercept)

# Show top positive and negative influential features
print("\nTop Influential Features:\n")
print(coef_df.to_string(index=False))

Intercept: -4.156678040680314

Top Influential Features:

          Feature  Coefficient
 poutcome_indexed     0.664650
  housing_indexed     0.467824
  marital_indexed     0.179996
    month_indexed     0.121712
         duration     0.003612
              age     0.000000
          balance     0.000000
              day     0.000000
      job_indexed     0.000000
         previous     0.000000
            pdays     0.000000
         campaign     0.000000
  default_indexed     0.000000
education_indexed     0.000000
  contact_indexed    -0.064667
     loan_indexed    -0.468615


Feature importance analysis revealed that the most influential predictor was the previous campaign outcome (poutcome). Clients with housing loans and those contacted during specific months showed a higher likelihood of subscribing to a term deposit. In contrast, clients with existing personal loans were less likely to subscribe. Features such as age, job, and account balance had relatively little impact on the model's predictions, indicating that campaign-related factors were more important than demographic characteristics.